# Convergence of biotx

The manufactured-solution studies of `experiments/convergence/`, in space and in time. Run the notebook from this folder.
Each study is computed once and saved to `results/<name>.npz`; later runs only load it (set `RERUN = True` to compute again).
Space studies: $L^2$ and $H^1$-seminorm panels. Time studies: $L^2$ only, because there both norms converge at the time order. The triangles show the expected orders; the legend shows the observed order of the last refinement.

In [10]:
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.ticker import NullFormatter
from biotx import convergence as cv

RERUN = True
# name: (study file, overrides, expected L2 order per plotted field; in space the H1 seminorm is one less)
STUDIES = {
    "biot2 P1/P1, space":       ("mms_biot2_space.yaml", {}, {"d": 2, "p": 2}),
    "biot2 P2/P2, space":       ("mms_biot2_space.yaml", {"discretization.deg": 2, "time.dt": 0.0015625,
                                                          "convergence.n": [4, 8, 16, 32]}, {"d": 3, "p": 3}),
    "biot3 P2/P2/P1, space":    ("mms_biot3_space.yaml", {}, {"d": 3, "w": 3, "p": 2}),
    "biot2 Crank-Nicolson, time": ("mms_biot2_time.yaml", {}, {"d": 2, "p": 2}),
    "biot2 backward Euler, time": ("mms_biot2_time.yaml", {"time.theta": 1.0}, {"d": 1, "p": 1}),
    "biot3 Newmark, time":      ("mms_biot3_time.yaml", {}, {"d": 2, "w": 2, "p": 2}),
}
COLORS = {"d": "#2a78d6", "p": "#eb6834", "w": "#1baf7a"}
MARKERS = {"d": "o", "p": "s", "w": "^"}

In [11]:
def get(title):
    file, overrides, _ = STUDIES[title]
    name = title.replace(" ", "_").replace(",", "").replace("/", "")
    path = Path("results") / f"{name}.npz"
    if RERUN or not path.exists():
        res = cv.run(cv.load(file, {**overrides, "output.name": name}))
        print(res.table())
        res.save(path)
    return cv.load_result(path)


def triangle(ax, s, y, order, below=True):
    """Slope triangle of the given order, one refinement wide, a factor 3 below (or above) y."""
    x0, x1 = s[-1], s[-1] * 2
    y0 = y / 3 if below else y * 3
    y1 = y0 * 2**order
    if below:                                       # right angle under the hypotenuse
        ax.plot([x0, x1, x1, x0], [y0, y0, y1, y0], color="0.35", lw=1)
        ax.annotate(f"{order}", (x1, (y0 * y1) ** 0.5), xytext=(-5, 0), textcoords="offset points",
                    ha="right", va="center", color="0.25")
    else:                                           # right angle above it
        ax.plot([x0, x1, x0, x0], [y0, y1, y1, y0], color="0.35", lw=1)
        ax.annotate(f"{order}", (x0, (y0 * y1) ** 0.5), xytext=(5, 0), textcoords="offset points",
                    ha="left", va="center", color="0.25")


def plot(title):
    res, expected = get(title), STUDIES[title][2]
    time = res["experiment"]["convergence"]["study"] == "time"
    s = res["dt"] if time else res["h"]
    norms = ["L2"] if time else ["L2", "H1"]       # in time both norms converge at the same order
    fig, axes = plt.subplots(1, len(norms), figsize=(5 * len(norms), 4), squeeze=False)
    for ax, norm in zip(axes[0], norms):
        finest = {}                                 # expected order -> lowest finest-level error
        for f, order in expected.items():
            e = res[f"{f}_{norm}"]
            ax.loglog(s, e, marker=MARKERS[f], color=COLORS[f], lw=1.6,
                      label=f"{f}  ({res[f'rate_{f}_{norm}'][-1]:.2f})")
            k = order - (norm == "H1")              # space: one order less in the H1 seminorm
            finest[k] = min(finest.get(k, e[-1]), e[-1])
        lowest = min(finest.values())
        for k, y in finest.items():                 # one per expected order: below the lowest lines,
            triangle(ax, s, y, k, below=(y == lowest))   # above the others
        ax.invert_xaxis()                           # refinement to the right
        ax.xaxis.set_minor_formatter(NullFormatter())
        ax.set(xlabel="$\\Delta t$ [s]" if time else "$h$", title=f"{norm} {'seminorm ' if norm == 'H1' else ''}error")
        ax.grid(True, which="both", color="0.9")
        ax.legend(title="field  (last order)", frameon=False)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()

In [ ]:
for title in STUDIES:
    plot(title)